# Reinforcement learning with verifiable rewards

Supervised fine-tuning is easy to explain: show the model the right answer and make that answer more likely. But for a lot of the problems we care about, nobody has written the right answer down. What we do have is a way to check one. Code either passes its tests or it doesn't, and a maths answer either matches or it doesn't.

A check is enough to learn from. Let the model try a few times, check each try, and nudge it towards the ones that passed. That's reinforcement learning with verifiable rewards, RLVR for short, and it's the recipe behind DeepSeek-R1's reasoning ([DeepSeek-AI, 2025](https://arxiv.org/abs/2501.12948)).

![The RLVR loop. The policy writes four replies to a GSM8K question, a checker compares each boxed answer with the reference, 72, and the update makes the replies that scored above the group's mean more likely.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-loop.svg)

That picture is the whole method. We'll build it with GRPO, the variant DeepSeekMath introduced ([Shao et al., 2024](https://arxiv.org/abs/2402.03300)) and R1 was trained with, and use it to teach Qwen2.5-0.5B-Instruct grade-school maths with Dew's `GRPOObjective`.

## Setup

In [1]:
%pip install -q "dewml[cuda13,interop,streaming] @ git+https://github.com/AshishKumar4/dew@bb6585ae3599f4f3bb46dbcd1291273d7f167165"

In [2]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPTS = 16          # questions per batch
GROUPS = 8            # replies per question, G
PROMPT_TOKENS = 256   # the longest question we keep, in tokens
NEW_TOKENS = 512      # the longest reply
STEPS = 200           # batches of replies to train on
LEARNING_RATE = 3e-6
BETA = 0.04           # weight of the KL penalty
SEED = 0

In [3]:
import json
from dataclasses import replace
from types import SimpleNamespace

import dew  # before JAX's backend opens, so its numerical settings apply
import jax
import jax.numpy as jnp
import numpy as np
from datasets import load_dataset

from dew import LocalTracker, Trainer, figures
from dew.config import OptimConfig
from dew.config.data import PromptData, loader
from dew.data import HFTokenizer, Loading
from dew.data.dataset import DataPartition
from dew.data.prompts import INFO_KEY
from dew.interop import PretrainedDecoder
from dew.nn.backbones.decoder_block import remat_policy
from dew.objectives.base import Step
from dew.objectives.rl import GRPOObjective, SampledRollout
from dew.rl import clipped_surrogate_terms, k3_kl
from dew.rl.sandbox import MathReward
from dew.sampling import Sampling
from dew.training import prepare_process

np.set_printoptions(precision=3, suppress=True)
prepare_process()  # Dew's setup for a training process, before the first JAX call

## A task we can check

GSM8K ([Cobbe et al., 2021](https://arxiv.org/abs/2110.14168)) is 8,500 grade-school word problems, each with a worked solution that ends in a single number. We only need that number. We'll ask the model to show its working and put the final answer in `\boxed{}`, and Dew's `MathReward` gives a reply 1 if its last boxed answer equals the reference and 0 otherwise. It compares numbers exactly, so `72`, `72.0` and `\frac{144}{2}` all count.

Here are three replies to the first training question, and what the checker makes of them.

In [4]:
gsm8k = load_dataset("openai/gsm8k", "main")
train, test = gsm8k["train"], gsm8k["test"]


def reference(row):
    # A worked solution ends with "#### 72"; the number is all we keep.
    return row["answer"].split("####")[-1].strip().replace(",", "")


checker = MathReward()
question = train[0]
print(f"{len(train)} training and {len(test)} test questions\n")
print(question["question"], "\nreference:", reference(question), "\n")
for reply in [r"In May she sold 48 / 2 = 24, so \boxed{72} in all.",
              r"She sold 48 * 2 = 96 in May, so \boxed{144}.",
              r"She sold 48 + 24 = 72 clips."]:
    print(f"{checker('gsm8k', reply, reference(question), ''):.0f}  {reply}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


7473 training and 1319 test questions

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? 
reference: 72 

1  In May she sold 48 / 2 = 24, so \boxed{72} in all.
0  She sold 48 * 2 = 96 in May, so \boxed{144}.
0  She sold 48 + 24 = 72 clips.


The first reply is right. The second boxes the wrong number. The third gets the arithmetic right but never boxes it, so the checker can't find an answer and gives it 0. That's harsh, but it's what makes the check trustworthy: it never has to guess which number in a reply is the answer.

## The model before training

We start from Qwen2.5-0.5B-Instruct ([Qwen Team, 2024](https://arxiv.org/abs/2412.15115)), a small model that already follows instructions and can do some arithmetic. Before we train it, let's grade it on all 1,319 test questions, which it will never see during training. It answers greedily, always taking its most likely token, so the same question always gets the same reply.

In [5]:
source = PretrainedDecoder.load(MODEL, dtype=jnp.bfloat16, param_dtype=jnp.float32,
                                max_seq_len=PROMPT_TOKENS + NEW_TOKENS)
tokenizer = HFTokenizer(MODEL)
stock = source.text_generation().sampling
ends = dict(eos_token_ids=stock.eos_token_ids, pad_token_id=stock.pad_token_id)
INSTRUCTION = r"Solve the problem step by step, then give the final answer in \boxed{}."


def messages(row):
    return [{"role": "user", "content": f"{row['question']}\n{INSTRUCTION}"}]


def chat(row):
    # The question as the model's chat template writes it, up to where the reply starts.
    return tokenizer.tokenizer.apply_chat_template(messages(row), tokenize=False,
                                                   add_generation_prompt=True)


def grade(variables, rows, batch=128):
    # Every reply to `rows` from the model with these weights, and which ones are right.
    task = replace(source, variables=variables).text_generation(
        sampling=Sampling(temperature=0.0, **ends))
    prompts = [chat(row) for row in rows]
    replies = []
    for start in range(0, len(prompts), batch):
        replies += task(prompts[start:start + batch], NEW_TOKENS, key=SEED).text
    right = [checker("gsm8k", reply, reference(row), "")
             for reply, row in zip(replies, rows)]
    return replies, np.asarray(right)


before, right_before = grade(source.variables, test)
print(f"{right_before.mean():.1%} of the {len(test)} test questions right\n")
print(test[0]["question"], "\nreference:", reference(test[0]), "\n")
print(before[0])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

48.3% of the 1319 test questions right

Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market? 
reference: 18 

To determine how much Janet makes at the farmers' market every day, we need to follow these steps:

1. Calculate the total number of eggs laid by the ducks in a day.
2. Determine the number of eggs sold at the farmers' market.
3. Calculate the revenue from selling the eggs at the farmers' market.

**Step 1: Calculate the total number of eggs laid by the ducks in a day.**

Janet's ducks lay 16 eggs per day. Therefore, the total number of eggs laid in a day is:
\[ 16 \text{ eggs/day} \]

**Step 2: Determine the number of eggs sold at the farmers' market.**

Janet eats 3 eggs for breakfast every morning and bakes 4 muffins for her friends every day. Therefo

Before any training it gets 48.3% of the test questions right. Its reply to the first one shows what we're starting from: it lays the problem out in steps, works through them, and boxes 18, which is right. It doesn't always go this well. On about half the questions it slips somewhere along the way, and on a few it never gets to a boxed answer at all. That's the sweet spot for RL. The model can already do the task some of the time, so when we sample several replies, some of them will pass.

## Why nudging towards the right replies works

How should a check, which only says yes or no, change the weights? Let's shrink the problem as far as it goes. Say the model had to pick one of five answers, and only one is right. Its policy is a softmax over five numbers, $\pi_\theta = \mathrm{softmax}(\theta)$, and we'd like to raise the expected reward $J(\theta) = \sum_a \pi_\theta(a)\, R(a)$.

But a checker has no gradient; you can't differentiate "the boxed number was 72". There's a neat way around that. Because $\nabla \pi = \pi\, \nabla \log \pi$, the gradient of the expected reward becomes an average over the model's own tries:

$$
\nabla J = \mathbb{E}_{a \sim \pi_\theta}\big[R(a)\, \nabla \log \pi_\theta(a)\big].
$$

Read it out loud: sample an answer, look at its reward, and push its log-probability up by that much. That's REINFORCE ([Williams, 1992](https://link.springer.com/article/10.1007/BF00992696)), and the rest of this notebook is refinements of this one line. For the softmax the push is $\nabla_\theta \log \pi_\theta(a) = \mathrm{onehot}(a) - \pi_\theta$, which raises the sampled answer and lowers every answer in proportion to how likely it already was. A reply is the same kind of choice, made one token at a time.

The catch is that word "average". A reply costs hundreds of tokens to generate, so we only get a handful per question, and a handful of scores is a noisy estimate. If none of them is right, there's nothing to push up at all. No trick recovers a signal from tries that all failed, which is why RLVR needs questions the model sometimes gets right. But even when some tries do pass, comparing each try against its siblings tames a lot of the noise.

## Compare each reply with its group

![Eight replies to one question scored 1 or 0, their mean, and the advantages. Right replies get a positive advantage and are pushed up, wrong ones get a negative one and are pushed down.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-advantage.svg)

Say a reply scores 1. How much should it go up? That depends on the question. On a question the model almost always gets right, one more right reply tells us little, and on one it usually fails, a right reply is exactly what we want more of. So GRPO samples a group of $G$ replies to the same question and judges each against the others, as in the picture: eight scores, their mean, and how far each one sits from the mean in units of the group's spread.

$$
A^{(i)} = \frac{R^{(i)} - \operatorname{mean}_j R^{(j)}}{\operatorname{std}_j R^{(j)} + 10^{-6}}.
$$

Beat your group and you're pushed up, fall short and you're pushed down, and dividing by the spread puts easy and hard questions on one scale. When every reply in a group scores the same, all right or all wrong, every advantage is zero and the question teaches nothing this time. (The $10^{-6}$ only keeps us from dividing by zero.) Subtracting the mean doesn't change which way the gradient points on average, since $\mathbb{E}_{a \sim \pi}[\nabla \log \pi(a)] = \nabla \sum_a \pi(a) = 0$, but it does make the estimate less noisy.

`SampledRollout` asks the model for `GROUPS` replies to each question, runs the checker on all of them, and packs a batch `GRPOObjective` can train on. Here's one real group: the eight replies our untrained model wrote to one training question, each shown by its last few tokens.

In [6]:
def record(index, row):
    # A question as a prompt record: the chat, the reference the checker compares with,
    # and the question's index, which reaches the checker as its extra information.
    return {"prompt": messages(row), "data_source": "gsm8k",
            "reward_model": {"ground_truth": reference(row)},
            "extra_info": {"index": index}}


width = PROMPT_TOKENS + NEW_TOKENS
# Full recomputation in the backward pass keeps a step's activations small.
policy = source.model.clone(remat=remat_policy("full"))
# With beta > 0 the objective keeps a frozen copy of the starting weights for the KL.
objective = GRPOObjective(policy, width - 1, variables=source.variables, beta=BETA)
sampling = Sampling(temperature=1.0, **ends)
rollout = SampledRollout(objective, checker, decode=tokenizer.decode, groups=GROUPS,
                         max_new_tokens=NEW_TOKENS, sampling=sampling)
records = tuple(record(index, row) for index, row in enumerate(train))
data = loader(PromptData(source=records, tokenizer=MODEL, max_prompt_len=PROMPT_TOKENS,
                         pad_id=stock.pad_token_id),
              batch=PROMPTS, read="memory", loading=Loading(workers=0))
questions = next(iter(data.train(DataPartition())))
start = SimpleNamespace(variables=source.variables, updates=0)
batch = rollout(start, questions, jax.random.key(SEED))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

In [7]:
# The batch packs replies into rows, several to a row where they fit. session_index
# says which reply a token belongs to, and replies q * GROUPS to q * GROUPS + GROUPS - 1
# answer the batch's question q.
def written(s):
    return (batch["session_index"] == s) & (batch["response_mask"] > 0)


sessions = range(PROMPTS * GROUPS)
by_question = np.array([batch["advantages"][written(s)][0] for s in sessions])
by_question = by_question.reshape(PROMPTS, GROUPS)
# The first question whose replies didn't all score the same.
q = next(q for q in range(PROMPTS) if np.ptp(by_question[q]) > 0)
group, advantages = range(q * GROUPS, (q + 1) * GROUPS), by_question[q]
# A finished reply ends with the end-of-turn token, which the checker never sees.
replies = [[token for token in batch["input_ids"][written(s)].tolist()
            if token not in stock.eos_token_ids] for s in group]
info = bytes(np.asarray(questions[INFO_KEY][q], np.uint8)).rstrip(b"\0")
asked = train[json.loads(info)["index"]]
rewards = np.array([checker("gsm8k", tokenizer.decode(reply), reference(asked), "")
                    for reply in replies])
print(asked["question"], "\nreference:", reference(asked))

tail = [[tokenizer.decode([token]).strip() or "·" for token in reply[-8:]]
        for reply in replies]
shade = ["accent_fill" if a > 0 else "warm_fill" for a in advantages]
figures.tokens(tail, fills=[[fill] * len(row) for fill, row in zip(shade, tail)],
               notes=[f"R {r:.0f}  A {a:+.2f}" for r, a in zip(rewards, advantages)])

A math teacher had $100 to buy three different types of calculators. A basic calculator costs $8. A scientific calculator costs twice the price as the basic while a graphing calculator costs thrice the price as the scientific. How much change did she receive after buying those three different types of calculators? 
reference: 28


Figure(611.8 x 282, 8 shapes)

Five of the eight replies box 28, the reference, and score 1. Of the other three, one boxes 0, one boxes 18, and one runs out of room in the middle of a sum. With 5 of 8 right the group's mean is 0.625, so a right reply sits +0.72 above it in units of the spread and a wrong one −1.21 below. Notice the asymmetry: on a question the model mostly gets right, a miss gets pushed down harder than a hit gets pushed up. On a question it mostly fails, it would be the other way round.

## From one choice to a whole reply

![One reply as the loss reads it: the end of the question with mask 0, and the tokens the policy wrote, 48 + 24 = 72, with mask 1, each carrying the reply's advantage.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-sequence.svg)

A reply is a sequence of choices, one per token, and its few hundred tokens have to share one reward. Because a reply's log-probability is the sum of its tokens' log-probabilities, we give every token the model wrote the same advantage, its reply's, as in the picture. The model didn't choose the question, and a mask $m$ (1 on written tokens, 0 everywhere else) leaves those tokens out. With $\ell_t = \log \pi_\theta(y_t \mid q, y_{<t})$ for one token, Dew minimizes the average over every written token in the batch:

$$
L(\theta) = -\frac{\sum_{i,t} m^{(i)}_t A^{(i)} \ell^{(i)}_t}{\sum_{i,t} m^{(i)}_t}.
$$

The next cell checks Dew against what we've described, on the group above. It recomputes the advantages from the checker's scores in NumPy. Then it writes $L$ itself, from the model's own next-token probabilities, and compares its gradient with the gradient of Dew's loss. At the starting weights the ratio and the KL penalty of the next two sections change nothing, so the two should agree.

In [8]:
ours = (rewards - rewards.mean()) / (rewards.std(ddof=1) + 1e-6)
np.testing.assert_allclose(advantages, ours, rtol=1e-5, atol=1e-6)
print("the advantages match")

the advantages match


In [9]:
# The rows that hold the group, with the loss kept to its replies, and every written
# token rescored at the starting weights, so each ratio starts at exactly 1.
held = np.isin(batch["session_index"], group)
rows = {name: column[held.any(axis=1)] for name, column in batch.items()}
rows["response_mask"] = rows["response_mask"] * held[held.any(axis=1)]
rows["old_log_probs"] = np.asarray(objective.packed_log_probs(source.variables, rows))
step = Step(step=jnp.zeros((), jnp.int32), key=jax.random.key(0), ema=source.variables)


def dews_loss(params):
    statistics, _ = objective.loss(params, rows, step)
    return statistics.mean()[0]


def our_loss(params):
    tokens = jnp.asarray(rows["input_ids"])
    m, a = jnp.asarray(rows["response_mask"]), jnp.asarray(rows["advantages"])
    packing = {"segment_ids": jnp.asarray(rows["text_segment_ids"])[:, :-1],
               "positions": jnp.asarray(rows["text_positions"])[:, :-1]}
    logits = policy.apply(params, tokens[:, :-1], **packing).astype(jnp.float32)
    log_probs = jax.nn.log_softmax(logits)
    ell = jnp.take_along_axis(log_probs, tokens[:, 1:, None], axis=-1)[..., 0]
    return -jnp.sum(m[:, 1:] * a[:, 1:] * ell) / jnp.sum(m[:, 1:])


def relative_gap(a, b):
    # How far apart two gradients are, as a share of the second one's length.
    a, b = jax.tree.leaves(a), jax.tree.leaves(b)
    gap = sum(float(jnp.sum((x - y) ** 2)) for x, y in zip(a, b))
    return (gap / sum(float(jnp.sum(y ** 2)) for y in b)) ** 0.5


gap = relative_gap(jax.grad(dews_loss)(source.variables),
                   jax.grad(our_loss)(source.variables))
assert gap < 0.02
print(f"the gradients match, to {gap:.2%} of the gradient's length")

the gradients match, to 1.04% of the gradient's length


Both match. The gradients agree to about 1% of the gradient's length, which is what bfloat16 leaves. The model computes in bfloat16, which keeps only two or three significant digits, and Dew's loss and ours do the same sums in a different order, so they round differently.

## Taking more than one step per batch

![The clipped objective against the ratio. With a positive advantage it rises until the ratio reaches 1.2 and is flat after; with a negative one it is flat below 0.8 and past the dual clip at 3. Where it is flat, the clipped term gives the token no gradient.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-clip.svg)

Generating replies is the slow part, so we'd like more than one gradient step per batch, or to train on replies a separate engine wrote while the last update ran. Either way the model has moved since it wrote the batch. Can we still train on it?

PPO ([Schulman et al., 2017](https://arxiv.org/abs/1707.06347)) weights each token by the ratio $\rho_t = \pi_\theta(y_t) / \pi_\text{old}(y_t)$, how much more or less likely the current model finds it than the model that wrote it, and stops pushing once that ratio has moved far enough the way the advantage asks:

$$
\max\big(-\rho_t A,\; -\operatorname{clip}(\rho_t, 0.8, 1.2)\, A\big).
$$

In the picture, a positive advantage keeps improving the objective until the ratio reaches 1.2. Past that the line is flat, so while the ratio stays above 1.2 the token gets no gradient from the reward term. A negative advantage does the same below 0.8. For negative advantages Dew also caps the term at $-3A$, the dual clip of [Ye et al., 2020](https://arxiv.org/abs/1912.09729), so one token with a runaway ratio can't take over a step. Here are Dew's clipped terms at a few ratios, next to the formula.

In [10]:
ratios = np.array([0.5, 0.9, 1.1, 1.5, 4.0])
for advantage in (1.0, -1.0):
    formula = np.maximum(-ratios * advantage, -np.clip(ratios, 0.8, 1.2) * advantage)
    if advantage < 0:
        formula = np.minimum(formula, -3.0 * advantage)
    terms, _ = clipped_surrogate_terms(jnp.log(ratios)[None], jnp.asarray([advantage]),
                                       jnp.ones((1, ratios.size)))
    np.testing.assert_allclose(terms[0], formula, rtol=1e-6)
    print(f"A = {advantage:+.0f}: terms {np.asarray(terms[0])} at ratios {ratios}")

A = +1: terms [-0.5 -0.9 -1.1 -1.2 -1.2] at ratios [0.5 0.9 1.1 1.5 4. ]
A = -1: terms [0.8 0.9 1.1 1.5 3. ] at ratios [0.5 0.9 1.1 1.5 4. ]


The terms follow the formula. With a positive advantage they stop at −1.2 past a ratio of 1.2, and with a negative one they sit at 0.8 below 0.8 and stop at 3 when the dual clip takes over.

## Keeping the model close to where it started

![The k3 estimate of the KL divergence against the policy's probability over the reference's. It is zero where they agree and rises on both sides, so its gradient pulls a drifted token back.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-kl.svg)

GRPO's loss has one more term. While it chases the reward, the model can drift far from where it started and lose what it knew, so GRPO adds $\beta$ times a per-token estimate of the KL divergence from the starting policy $\pi_\text{ref}$:

$$
k_3 = \frac{\pi_\text{ref}}{\pi_\theta} - \log \frac{\pi_\text{ref}}{\pi_\theta} - 1 \;\ge 0.
$$

It's zero when the two policies agree and grows as the model drifts either way, so its gradient points back towards the start, like the ball in the picture. Where the clip has flattened the reward term, this one can still pull, unless it has reached its own cap. Dew caps it at 10, as verl does, so one token the model has nearly ruled out can't swamp the loss; the last pair in the next cell is past that cap.

In [11]:
policy_probs = np.array([0.1, 0.3, 0.5, 0.9, 0.01])
reference_probs = np.array([0.3, 0.3, 0.25, 0.5, 0.2])
# Dew caps k3 at 10, and the log-ratio inside it at ±20, as verl does.
d = np.clip(np.log(reference_probs / policy_probs), -20, 20)
estimate = k3_kl(jnp.log(policy_probs), jnp.log(reference_probs))
capped = np.minimum(np.exp(d) - d - 1, 10)
np.testing.assert_allclose(estimate, capped, rtol=1e-5, atol=1e-7)
print("formula:", np.exp(d) - d - 1)
print("k3_kl:  ", np.asarray(estimate))

formula: [ 0.901  0.     0.193  0.143 16.004]
k3_kl:   [ 0.901  0.     0.193  0.143 10.   ]


The first four values match the formula. For the last pair the formula gives 16.0, and Dew stops at the cap of 10.

## Training it

Now we put the pieces together. Every step draws `PROMPTS` fresh questions, samples and scores `GROUPS` replies to each, and takes one Adam step on the batch, with `GRPOObjective` computing everything above. The tracker records each batch's mean reward and the share of replies that ran out of room before they finished.

In [12]:
optimizer = OptimConfig(learning_rate=LEARNING_RATE, b2=0.99, weight_decay=0.0,
                        clip_grads=1.0)
trainer = Trainer(objective, optimizer, key=jax.random.key(SEED), rollout=rollout,
                  tracker=LocalTracker("runs/10-rlvr"))
state = trainer.fit(data, steps=STEPS, log_every=5)

with open("runs/10-rlvr/scalars.jsonl") as lines:
    logged = [json.loads(line) for line in lines]
logged = [row for row in logged if "rollout/reward/mean" in row["scalars"]]
figures.curve({"mean reward": [row["scalars"]["rollout/reward/mean"] for row in logged],
               "cut off": [row["scalars"]["rollout/status/truncated"] for row in logged]},
              x=[row["step"] for row in logged], x_label="step",
              y_label="share of replies", y_range=(0, 1), smooth=4)

Training CausalTransformer from step 0 to 200: 494.0M parameters, on 1 × NVIDIA A100-SXM4-40GB, batch 16, float32 parameters, bfloat16 compute

10:23:12 WARNING dew.nn.kernels.tuning: 6 kernel launches have no config measured on NVIDIA A100-SXM4-40GB, so they run the nearest measured shape's or their heuristic: measure them on a GPU to yourself with Trainer(tune_kernels=True) or `python tools/kernel_picks.py`


train/mfu is unavailable for GRPOObjective: GRPOObjective needs counts for its auxiliary model forwards

step 5/200  loss 0.01368  actor/pg_clipfrac 0.001148  actor/pg_clipfrac_lower 0  actor/ppo_kl 4.591e-04  kl 0.002183  mismatch/ess 1.000  mismatch/k3_kl 0  mismatch/kl 0  pg 0.01359  data_wait_ms 0.02624  grad_norm 0.8733  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 24.30  step_time_ms 14,795  samples_per_sec 8.651  hlo_flops_util_lower_bound 9.6%  hfu_lower_bound 10.8%  unsized_kernels 291.0  rollout_seconds 81.99  accepted 100.0%  learning_rate 3.000e-06  length/mean 312.9  reward/mean 0.4531  status/truncated 3.9%  accepted/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 7,472  attempted/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 7,473  chunks_per_record/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 0  rejected/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3/over_length 1.000  truncated/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 0  0:52:53 left

step 10/200  loss -0.004053  actor/pg_clipfrac 8.603e-04  actor/pg_clipfrac_lower 0  actor/ppo_kl 2.783e-04  kl 0.007696  mismatch/ess 1.000  mismatch/k3_kl 0  mismatch/kl 0  pg -0.004361  data_wait_ms 0.02811  grad_norm 0.9489  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 24.30  step_time_ms 19,739  samples_per_sec 6.484  hlo_flops_util_lower_bound 7.2%  hfu_lower_bound 8.1%  unsized_kernels 291.0  rollout_seconds 70.81  accepted 100.0%  learning_rate 3.000e-06  length/mean 263.4  reward/mean 0.5391  status/truncated 0.8%  accepted/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 7,472  attempted/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 7,473  chunks_per_record/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 0  rejected/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3/over_length 1.000  truncated/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 0  0:52:44 left

...

step 195/200  loss 0.01934  actor/pg_clipfrac 0.001015  actor/pg_clipfrac_lower 0  actor/ppo_kl 4.545e-04  kl 0.03573  mismatch/ess 1.000  mismatch/k3_kl 0  mismatch/kl 0  pg 0.01791  data_wait_ms 0.02940  grad_norm 0.7893  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 24.30  step_time_ms 16,947  samples_per_sec 7.553  hlo_flops_util_lower_bound 8.4%  hfu_lower_bound 9.4%  unsized_kernels 291.0  rollout_seconds 73.88  accepted 100.0%  learning_rate 3.000e-06  length/mean 269.5  reward/mean 0.7422  status/truncated 2.3%  accepted/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 7,472  attempted/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 7,473  chunks_per_record/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 0  rejected/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3/over_length 1.000  truncated/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 0  0:01:25 left

step 200/200  loss 0.04158  actor/pg_clipfrac 0.001292  actor/pg_clipfrac_lower 0  actor/ppo_kl 2.843e-04  kl 0.03641  mismatch/ess 1.000  mismatch/k3_kl 0  mismatch/kl 0  pg 0.04013  data_wait_ms 0.03074  grad_norm 0.8603  compiles 1.000  cache_hits 0  cache_misses 0  peak_memory_gib 24.30  step_time_ms 14,242  samples_per_sec 8.987  hlo_flops_util_lower_bound 10.0%  hfu_lower_bound 11.2%  unsized_kernels 291.0  rollout_seconds 74.27  accepted 100.0%  learning_rate 3.000e-06  length/mean 314.5  reward/mean 0.5156  status/truncated 7.0%  accepted/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 7,472  attempted/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 7,473  chunks_per_record/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 0  rejected/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3/over_length 1.000  truncated/memory:6bc361644c6a032d8f5b721f528fdd7ea56e0305448c8749d34237c0effa90e3 0

Trained 200 steps in 0:58:09: first step after 113.78 s, then 0.1 step/s

96.7% of the wall time in steps, final loss 0.04158

Figure(460 x 290, 24 shapes)

Each point is one batch of 16 questions, so the faint line jumps around a lot; the thick one is a running mean over four points. It climbs from about 0.50 to 0.59 over the 200 steps. The share of replies that ran out of room before finishing stays around 5% the whole way. But the training reward is measured on questions the model is learning from, sampled at temperature 1. The real test is the held-out questions, answered greedily, as before.

## The model after training

Now the same 1,319 test questions again, answered greedily by the trained weights.

In [13]:
after, right_after = grade(state.variables, test)
print(f"right before training {right_before.mean():.1%}, "
      f"after {right_after.mean():.1%}")


def outcomes(replies, right):
    unboxed = np.mean([r"\boxed{" not in reply for reply in replies])
    return [right.mean(), 1 - right.mean() - unboxed, unboxed]


figures.bars([outcomes(before, right_before), outcomes(after, right_after)],
             labels=["right", "wrong", "no answer"],
             captions=["before training", "after training"],
             colors=["accent", "warm", "tint1"], y_label="share of test replies")

right before training 48.3%, after 52.5%


Figure(460 x 260, 16 shapes)

Right answers go from 48.3% to 52.5% of the 1,319 test questions. Where did those four points come from? About half from replies that now end with a boxed answer (no answer drops from 6% to 4%), and half from fewer wrong ones (46% down to 44%). RL rewards both, and it can't tell them apart: a reply that forgot to box its answer scored 0, just like a wrong one.

In [14]:
fixed = next(i for i in range(len(test)) if right_after[i] > right_before[i])
print(test[fixed]["question"], "\nreference:", reference(test[fixed]))
print("\n--- before ---\n" + before[fixed])
print("\n--- after ---\n" + after[fixed])

A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take? 
reference: 3

--- before ---
To determine the total number of bolts of blue and white fiber needed, we start by defining the variables and setting up the equation based on the given information.

Let:
- \( B \) be the number of bolts of blue fiber.
- \( W \) be the number of bolts of white fiber.

According to the problem, the robe takes 2 bolts of blue fiber and half that much white fiber. Therefore, we can write the following equation:

\[ 2B = W \]

We need to find the total number of bolts, which is the sum of the blue and white fibers:

\[ B + W \]

To find \( B + W \), we can substitute \( W \) from the equation \( 2B = W \) into the total equation:

\[ B + W = 2B \]

Thus, the total number of bolts needed is:

\[
\boxed{2B}
\]

--- after ---
To determine the total number of bolts needed, we need to follow these steps:

1. Identify the amount of blue fiber and white fiber re

Here's one of the questions it now gets right. Before training, the model turned "half that much white fiber" into the equation $2B = W$, lost track of what it was solving for, and boxed $2B$, which isn't even a number. After training it reads the numbers straight off the question: 2 bolts of blue, half of that is 1 bolt of white, 3 in all. All the model ever got back during training was a 1 or a 0 per reply.

## What changes at a larger scale

The loss stays the same. A bigger model gets its logits from a deeper network, but `GRPOObjective` computes the same per-token terms we just checked. What grows is everything around it. Rewards like running a program's tests need a sandbox (`dew.rl.sandbox`). While the trainer updates the model, a separate engine (Dew's `Server`, vLLM or SGLang) generates replies, so they can be one update old by the time we train on them; the ratio from earlier corrects for that, and `behavior_importance` caps each token's weight. And long or multi-turn episodes are packed into rows with masks, which [tutorial 11](11-llms-in-an-agent-harness.ipynb) walks through.

[`examples/train_rlvr.py`](https://github.com/AshishKumar4/dew/blob/main/examples/train_rlvr.py) puts all of this together and trains a model to write Python programs that pass their tests. [Post-training](../docs/concepts/post_training.md) covers Dew's other objectives.